In [153]:
# import json

# import numpy as np
# import osmnx as ox
import pandas as pd
import geopandas as gpd
from shapely import wkt

# import lts_functions
import LTS_OSM
# import LTS_plot

import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

CRS = 'EPSG:2805'
AREA_CONVERT = 3.861022e-7 # Convert from m^2 to mi^2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [123]:
# Load LTS csv
gdf = LTS_OSM.read_lts_csv(r'data\GreaterBoston_4_all_lts.csv')
gdf.to_crs(CRS, inplace=True)

gdf['geometry_centroid'] = gdf.centroid
gdf['geometry_way'] = gdf['geometry']

# Reduce bi-directional line pairs
gdf.drop_duplicates(subset=['osmid', 'geometry_centroid'], inplace=True)

In [124]:
geojsonFolder = '../osm-bikeparking/'

city = gpd.read_file(f'{geojsonFolder}data_cities.geojson')
hood = gpd.read_file(f'{geojsonFolder}data_neighborhoods.geojson')
council = gpd.read_file(f'{geojsonFolder}data_council_boston.json')

city.to_crs(CRS, inplace=True)
hood.to_crs(CRS, inplace=True)
council.to_crs(CRS, inplace=True)

# Pre-process city data
city['population'] = city['population'].str.replace(r'\D', '', regex=True).astype(float) # Remove non-digit values from populations, e.g. commas
city.rename(columns={'name': 'city'}, inplace=True)
city['population'] = city['population'].fillna(-1)
city = city.astype({"population": "int"})
city['area_sqmi'] = city.area * AREA_CONVERT # https://spatialreference.org/ref/epsg/2805/

# Pre-process neighborhood data
hood.rename(columns={'name': 'neighborhood'}, inplace=True)
hood['area_sqmi'] = hood.area * AREA_CONVERT # https://spatialreference.org/ref/epsg/2805/

hoodCentroid = hood[['neighborhood', 'geometry']]
hoodCentroid['geometry'] = hood.centroid
hoodCentroid = hoodCentroid.sjoin(city[['city','geometry']], how='inner', predicate='intersects')
hoodCentroid.set_index('neighborhood', inplace=True)
hood = hood.join(hoodCentroid['city'], on='neighborhood')
# hood = hood.join(hoodCentroid['city'], on='neighborhood', lsuffix='suf')

# Pre-process city council districts data
council['area_sqmi'] = council.area * AREA_CONVERT # https://spatialreference.org/ref/epsg/2805/

In [54]:
for x in gdf.columns:
    print(x)

u
v
key
osmid
highway
lanes
maxspeed
name
width
surface
oneway
bicycle
parking:left
parking:right
parking:right:orientation
lanes:forward
lanes:backward
turn:lanes:forward
service
sidewalk:right
sidewalk:left
lane_markings
parking:left:orientation
cycleway:right
cycleway:left
sidewalk
maxspeed:type
cycleway:right:lane
shoulder
cycleway:right:buffer
busway
cycleway:left:buffer
busway:left
turn:lanes:backward
foot
motor_vehicle
start_date
footway
crossing
crossing:signals
expressway
bridge
layer
turn:lanes
incline
covered
construction
cycleway:left:lane
crossing:markings
segregated
access
alt_name
cycleway:right:oneway
junction
oneway:bicycle
cycleway:left:oneway
parking:condition:right
parking:lane:left:parallel
bus:lanes:conditional
turn
sidewalk:right:surface
oneway:conditional
parking:condition:left
trolley_wire
tunnel
busway:right
official_name
placement
short_name
crossing:island
opening_date
footway:surface
bus
trolleybus
informal
parking:lane:right:parallel
traffic_calming
traffi

In [151]:
gdfSmall = gdf[['osmid', 'geometry_centroid', 'geometry_way', 'LTS_fwd', 'LTS_rev', 'geometry', 'highway']]
gdfSmall['length'] = gdfSmall.length
gdfSmall['geometry'] = gdfSmall['geometry_centroid']


gdfCity = gdfSmall.sjoin(city[['city','geometry']], how='inner', predicate='intersects')
gdfHood = gdfSmall.sjoin(hood[['neighborhood','geometry']], how='inner', predicate='intersects')
gdfCouncil = gdfSmall.sjoin(council[['LONGNAME', 'Councilor','geometry']], how='inner', predicate='intersects')

In [172]:

sumFwd = gdfCity.groupby(['city', 'LTS_fwd', 'highway'])['length'].sum().rename_axis(['city', 'LTS', 'highway'])
sumRev = gdfCity.groupby(['city', 'LTS_rev', 'highway'])['length'].sum().rename_axis(['city', 'LTS', 'highway'])
sumLTS = sumFwd + sumRev
sumLTS.loc[['Cambridge','Boston','Somerville','Brookline']].to_csv('total_city.csv')

sumFwd = gdfHood.groupby(['neighborhood', 'LTS_fwd', 'highway'])['length'].sum().rename_axis(['neighborhood', 'LTS', 'highway'])
sumRev = gdfHood.groupby(['neighborhood', 'LTS_rev', 'highway'])['length'].sum().rename_axis(['neighborhood', 'LTS', 'highway'])
sumLTS = sumFwd + sumRev
sumLTS.to_csv('total_neighborhood.csv')

sumFwd = gdfCouncil.groupby(['LONGNAME', 'Councilor', 'LTS_fwd', 'highway'])['length'].sum().rename_axis(['District', 'Councilor', 'LTS', 'highway'])
sumRev = gdfCouncil.groupby(['LONGNAME', 'Councilor', 'LTS_rev', 'highway'])['length'].sum().rename_axis(['District', 'Councilor', 'LTS', 'highway'])
sumLTS = sumFwd + sumRev
sumLTS.to_csv('total_council.csv')
sumLTS = sumLTS.reset_index()

gdfCouncilFilter = gdfCouncil[gdfCouncil['highway']!='residential']
sumFwd = gdfCouncilFilter.groupby(['LONGNAME', 'Councilor', 'LTS_fwd'])['length'].sum().rename_axis(['District', 'Councilor', 'LTS'])
sumRev = gdfCouncilFilter.groupby(['LONGNAME', 'Councilor', 'LTS_rev'])['length'].sum().rename_axis(['District', 'Councilor', 'LTS'])
sumLTS = sumFwd + sumRev
sumLTS = sumLTS.reset_index()
sumLTS = sumLTS[sumLTS['LTS']!='0.0']
sumLTS.to_csv('total_council_filter.csv')

In [173]:
sumLTS

,District,Councilor,LTS,length
1,District 1,Gabriela Coletta,1.0,179657.704201
2,District 1,Gabriela Coletta,2.0,130425.958465
3,District 1,Gabriela Coletta,3.0,73978.258312
4,District 1,Gabriela Coletta,4.0,13855.778553
6,District 2,Ed Flynn,1.0,104231.873475
7,District 2,Ed Flynn,2.0,97886.539121
8,District 2,Ed Flynn,3.0,76901.381478
9,District 2,Ed Flynn,4.0,6154.447381
11,District 3,John Fitzgerald,1.0,145918.119552
12,District 3,John Fitzgerald,2.0,183352.125119
